# 公式 o14 — The sGFRD Method（表面上の単分子反応拡散）

> **出典（E-Cell4 公式）**: Examples / example14 — https://ecell4.e-cell.org/examples/example14.html
>
> **sGFRD**（surface Green's Function Reaction Dynamics）は、eGFRD を **2 次元の曲面（膜）**へ拡張した
> 単分子解像度の反応拡散法。分子が膜面上を拡散して出会うまでの時間を、Green 関数で厳密に計算して飛ばす（イベント駆動）。
>
> ⚠️ **注記**: 出典ページは埋め込み画像/動画が大きく、本ノートでは**コードを自動取得できなかった**。
> 完全なコードは上記 URL を参照。ここでは **sGFRD の使い方（API の形）** をインストール済みモジュールで確認した範囲で示す。
> 空間・単分子で**重い**ため自動実行はしていない。

In [1]:
# インストール済み sgfrd モジュールの構成（実際に存在する API）
from ecell4_base import sgfrd
print('sgfrd exports:', [a for a in dir(sgfrd) if not a.startswith('_')])
# -> ['Factory', 'ReactionInfo', 'SGFRDFactory', 'SGFRDSimulator', 'SGFRDWorld', 'Simulator', 'World']

sgfrd exports: ['Factory', 'ReactionInfo', 'SGFRDFactory', 'SGFRDSimulator', 'SGFRDWorld', 'Simulator', 'World']


## 使い方の形（illustrative — 公式コードそのものではない）

他の空間ソルバ（o1 の egfrd, o12 の spatiocyte）と同じ **Factory → world → 分子追加 → simulator → run** の流れ。
sGFRD は曲面（球面など）上の反応拡散を扱う:

```python
from ecell4.prelude import *
from ecell4_base import sgfrd

with species_attributes():
    A | B | C | {'radius': 0.005, 'D': 1.0}
with reaction_rules():
    A + B == C | (ka, kd)
m = get_model()

f = sgfrd.Factory()                 # または sgfrd.SGFRDFactory()
w = f.world(Real3(L, L, L))         # 曲面（例: 球）を張った世界
w.bind_to(m)
w.add_molecules(Species('A'), N)
w.add_molecules(Species('B'), N)
sim = f.simulator(w)
sim.run(t_end)
```

## sGFRD が使える場面

- **膜上のシグナル伝達**（受容体・アダプターが 2D 膜面を拡散して出会う）を、混雑・離散性・拡散律速まで含めて正確に。
- eGFRD（3D 空間, o1 参照）の**曲面版**。分子が疎な系では、拡散待ち時間を解析的に飛ばせるので、
  格子法(Spatiocyte)より効率的なことがある。

**要点**: E-Cell4 は同じモデル記述のまま、**well-mixed（ODE/Gillespie）→ 区画(meso)→ 格子単分子(Spatiocyte)→ 連続空間単分子(eGFRD/sGFRD)**
とスケール/解像度を選べる。sGFRD はその「曲面・単分子」の端を担う。